# Paired evaluation, costs and learning curves
    No training is launched. The primary comparison uses the same fixed corrupted
    contexts for every agent. Never compare each agent against its own polluter.
    Report split minus unified from our runs; the paper's score is background context.
    Shared choices are recorded in docs/ASSUMPTIONS.md. Smoke/pilot values are
    diagnostics and cannot establish the full comparison's scientific conclusion.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, zipfile

candidates = [Path(os.environ.get("GASP_ROOT", ".")), Path.cwd(), Path.cwd().parent, Path("/content/gasp")]
ROOT = next((p.resolve() for p in candidates if (p / "src/gasp").is_dir()), None)
if ROOT is None:
    try:
        from google.colab import files
    except ImportError:
        raise RuntimeError("Unzip dist/gasp-colab.zip and open this notebook inside the project.")
    print("Select the delivered gasp-colab.zip archive.")
    uploaded = files.upload()
    archives = [p for p in uploaded if p.endswith(".zip")]
    if len(archives) != 1:
        raise ValueError("Select exactly one project archive")
    with zipfile.ZipFile(archives[0]) as z:
        for name in z.namelist():
            if not name.startswith("gasp/") or ".." in Path(name).parts:
                raise ValueError("Unexpected archive member")
        z.extractall("/content")
    ROOT = Path("/content/gasp")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

def command(*args):
    result = subprocess.run([sys.executable, *map(str,args)], cwd=ROOT,
                            env={**os.environ,"PYTHONPATH":str(ROOT/"src")},
                            text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout)
    result.check_returncode()

print("Project:", ROOT)


In [ ]:
# Optional persistence. Keep downloaded HF weights in Colab scratch; persist data/runs.
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive/gasp-experiment")
else:
    WORK = ROOT
WORK.mkdir(parents=True, exist_ok=True)
print("Data and outputs:", WORK)


In [ ]:
PROFILE = "full"
SEEDS = [42] if PROFILE=="pilot" else [42,52,62]
RUNS = WORK/f"runs/{PROFILE}-v4"
run_paths=[RUNS/f"{condition}-{seed}" for seed in SEEDS for condition in ("unified","split")]
from gasp.evaluation import aggregate
report=aggregate(run_paths,RUNS/"comparison",allow_pilot=PROFILE=="pilot")
print("Protocol:",report["protocol"])
print(json.dumps(report["summary"],indent=2))


In [ ]:
import pandas as pd
from IPython.display import display
rows=[]
for pair in report["per_seed"]:
    for metric in pair["unified"]:
        rows.append({"seed":pair["seed"],"metric":metric,
            **{c:100*pair[c][metric] if pair[c][metric] is not None else None
               for c in ("unified","split","split_minus_unified")}})
display(pd.DataFrame(rows))  # percentage points
costs=[]
for pair in report["per_seed"]:
    for condition in ("unified","split"):
        t=pair[f"cost_{condition}"]
        costs.append({"seed":pair["seed"],"condition":condition,
            "generated_tokens":sum(v["generated_tokens"] for v in t["rollouts"].values()),
            "training_seconds":t["training_wall_seconds"],
            "gpu_interval_seconds":t["cuda_elapsed_interval_seconds"],
            "peak_gpu_GiB":t["peak_gpu_allocated_bytes"]/2**30,
            "parameter_GiB":t["unique_role_parameter_bytes"]/2**30,
            "optimizer_GiB":t["optimizer_bytes"]/2**30})
display(pd.DataFrame(costs))


In [ ]:
import matplotlib.pyplot as plt
fig, axes=plt.subplots(1,2,figsize=(12,4))
for path in run_paths:
    history=[json.loads(x) for x in (path/"training.jsonl").read_text().splitlines()]
    tokens={0:0,**{h["step"]:sum(v["generated_tokens"] for v in h["telemetry"]["rollouts"].values()) for h in history}}
    metrics=[json.loads(p.read_text()) for p in sorted((path/"eval").glob("step-*/metrics.json"))]
    for metric,style in [("corrupted_initial_clean_solved","-"),("clean_full","--")]:
        y=[100*m["metrics"][metric] if m["metrics"][metric] is not None else float("nan") for m in metrics]
        axes[0].plot([m["step"] for m in metrics],y,style,label=f"{path.name} {metric}")
        axes[1].plot([tokens[m["step"]] for m in metrics],y,style,label=f"{path.name} {metric}")
axes[0].set_xlabel("Global optimizer updates");axes[1].set_xlabel("Actual training-related generated tokens")
for ax in axes:ax.set_ylabel("Accuracy (%)");ax.grid(alpha=.2)
axes[1].legend(fontsize=7)
fig.suptitle(f"{report['label']} — {report['protocol']}")
fig.tight_layout();fig.savefig(RUNS/"comparison/learning-curves.png",dpi=160)
plt.show()


In [ ]:
# Inspect paired disagreements instead of only looking at means.
path=RUNS/"comparison/disagreements.jsonl"
disagreements=[json.loads(x) for x in path.read_text().splitlines()]
display(pd.DataFrame(disagreements).head(20))


## Interpretation
    Report per-seed scores, means, sample standard deviations and paired differences.
    Three seeds give limited precision. Equal GRPO/repair rollout allocation does not guarantee
    equal actual tokens or GPU time; report both. Preparation costs are shared and
    stored in data manifests, evaluation costs in each metrics file.

    Published R1-1.5B GASP results provide context only: clean70.3±2.4,
    clean-solved recoverability95.6±2.2 (Table1), full-set recoverability78.5±1.2
    (Table2). Author artifacts and target-model runs are still needed before comparing numbers.
